In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data (from Week 4)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/promoter-gene-sequences/promoters.data"
df = pd.read_csv(url, header=None, names=['label', 'id', 'sequence'])

# Clean sequences
def clean_sequence(seq):
    return seq.strip().upper()

df['sequence'] = df['sequence'].apply(clean_sequence)
df['label'] = (df['label'] == '+').astype(float)

# One-hot encode
def one_hot_encode(seq):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    encoded = torch.zeros(len(seq), 4)
    for i, nucleotide in enumerate(seq):
        if nucleotide in mapping:
            encoded[i, mapping[nucleotide]] = 1.0
        # Unknown nucleotides remain as zeros
    return encoded

# Encode all sequences
sequences = [one_hot_encode(seq) for seq in df['sequence']]
sequences = torch.stack(sequences)  # (106, 57, 4)
labels = torch.tensor(df['label'].values, dtype=torch.float32).unsqueeze(1)  # (106, 1)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    sequences, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Sequence shape: {X_train[0].shape}")  # (57, 4)

In [ ]:
def decode(tensor: torch.Tensor) -> str:
        """
        Convert one-hot tensor back to DNA sequence.
        For validation/debugging.
        """
        index_to_nucleotide = {0: 'A', 1: 'C', 2: 'G', 3: 'T'}
        sequence = []
        for i in range(tensor.shape[0]):
            if torch.sum(tensor[i]) == 0:
                '''Handle unknown nucleotides (e.g., 'N') by adding 'N' to the sequence.'''
                sequence.append('N')
                continue
            index = torch.argmax(tensor[i]).item()
            nucleotide = index_to_nucleotide[index]
            sequence.append(nucleotide)

        return ''.join(sequence)
    

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import sys
sys.path.append('../src')  # Add parent directory to path
from Transformer import DNATransformerFull
# Train model
model = DNATransformerFull(vocab_size=4, d_model=64, num_heads=8)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

# Train for 50 epochs (same as before)
for epoch in range(50):
    model.train()
    for i in range(0, len(X_train), 8):
        batch_X = X_train[i:i+8]
        batch_y = y_train[i:i+8]
        
        optimizer.zero_grad()
        pred, _ = model(batch_X)
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_pred, _ = model(X_test)
            acc = ((test_pred > 0.5).float() == y_test).float().mean()
            print(f"Epoch {epoch+1}: Accuracy {acc:.2%}")

print("\n✅ Multi-head training complete!")

# Visualize EACH head's attention pattern
model.eval()
with torch.no_grad():
    _, attn_weights = model(X_test[:1])  # First test sequence
    # attn_weights: (1, 8, 57, 57)
    print(f"Attention weights shape: {attn_weights.shape}")
    print(f"Expected: (1, 8, 57, 57)")
    print(f"  batch=1, num_heads=8, seq_len=57, seq_len=57")

    # Check each dimension
    print(f"\nDimension breakdown:")
    print(f"  Batch: {attn_weights.size(0)}")
    print(f"  Heads: {attn_weights.size(1)}")
    print(f"  Query positions: {attn_weights.size(2)}")
    print(f"  Key positions: {attn_weights.size(3)}")

# Plot all 8 heads
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
nucleotides = list(decode(X_test[0].unsqueeze(1)))

for head in range(8):
    ax = axes[head // 4, head % 4]
    attn = attn_weights[0, head].cpu().numpy()  # (57, 57)
    
    im = ax.imshow(attn, cmap='Blues', aspect='auto')
    ax.set_title(f'Head {head}: Attention Pattern', fontweight='bold')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig('multihead_attention.png', dpi=150)
plt.show()